# project_16_humanization — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — humanization strategies, the trade-off, + the hello-world

**Standard slot:** *define & explore.* **For Project 16 this means:** understand the **non-human
therapeutic antibody** you will humanize and the **human germline frameworks** you will graft onto,
learn the three humanization strategies (**CDR grafting**, **resurfacing**, **germline-content
optimization**), fix the metrics table, and run the **mock** humanization hello-world end-to-end (D0).

Run `00_setup.ipynb` first in this session. Everything here runs with **no GPU** on the deterministic
`mock` backend. Compute for this whole project is genuinely **light** (humanness scoring + IgFold +
ΔΔG proxies are free-tier Colab **T4** friendly) — switch each `tool="mock"` to the real backend on
Colab when you are ready.

## The problem in one screen

A **non-human antibody** (murine, or a murine/human **chimera**) used as a therapeutic triggers an
**anti-drug-antibody (ADA)** response: the patient's immune system recognizes the non-human sequence as
foreign, clears the drug, and can cause adverse reactions. **Humanization** rewrites the antibody so it
looks like a **human germline antibody** (low ADA risk) while keeping the original **CDRs** that confer
binding. It is a **regulatory necessity** for non-human therapeutic candidates.

The three classic strategies:
- **CDR grafting** — transplant the non-human **CDR loops** onto a **human germline framework**
  (FR1..FR4). Maximally human framework, but the new framework residues that *support* the CDR loops
  (the **Vernier zone**) often perturb the loops → **affinity / stability loss**. The fix is selective
  **back-mutation** of Vernier-zone residues (restore the parental residue).
- **Resurfacing (veneering)** — keep the non-human framework **core**, mutate only the **surface-exposed**
  framework residues to human identity. Changes far fewer residues (lower ΔΔG risk) but achieves **less
  humanness**.
- **Germline-content optimization** — push the sequence toward the nearest human **germline** content,
  scored by humanness tools (OASis / Hu-mAb / T20 / AbLang).

**The central tension (this whole project):** *humanness ↔ stability is a trade-off.* More framework
humanization usually means more mutations means higher destabilization (**ΔΔG**). The deliverable is
**humanized variants + the trade-off analysis + the back-mutations needed + a validation plan** — NOT
"a humanized antibody". A computational design is a **hypothesis**; expression is not function; a
humanness score is not a guaranteed low-ADA outcome.

## The metrics table (what we will measure and filter on)

| Metric | Range | Means | Does **not** mean | Direction |
|--------|-------|-------|-------------------|-----------|
| humanness (OASis-like) | 0–1 | fraction of 9-mers matching human repertoire (proxy) | guaranteed low immunogenicity | higher = more human |
| humanness (T20-like) | 0–100 | rescaled human-likeness (proxy) | a clinical ADA prediction | higher = more human |
| germline_id | label | nearest human germline (proxy) | a validated germline call | human-like wanted |
| ΔΔG (proxy) | a.u. (signed) | predicted destabilization of the graft vs parental | a real FoldX/Rosetta kcal/mol | **lower** = more stable (>0 destabilizing) |
| n_framework_mutations | count | how many FR residues changed from parental | humanness by itself | trade-off knob |
| pLDDT (IgFold/AF2) | 0–100 | model local confidence of the Fv | stability / affinity | higher = more confident |
| scRMSD | Å | designed-vs-predicted backbone (self-consistency) | binding | ≤ 3.0 (antibody) |

The shared filter uses `filtering_pipeline.DEFAULT_CUTOFFS["antibody"]` (scRMSD ≤ 3.0, pLDDT ≥ 70,
pae_interaction ≤ 12); we map humanness and the ΔΔG proxy onto it in notebook 03. **Humanness and ΔΔG
here are TEACHING HEURISTICS, not the validated tools (OASis/Hu-mAb/T20/AbLang; FoldX/Rosetta)** — see
`MANUAL.md §2` and `humanization_tools.py`. Never report a heuristic number as a real result.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Pick your non-human antibody + human germline framework

These are the two fixed inputs to the campaign. **The student supplies a real published murine /
chimeric therapeutic antibody** (VH/VL sequence) and a **human germline framework** from IMGT/OAS — the
placeholders below are TEACHING sequences, **not** a specific real antibody. **Verify your antibody
sequence and your germline choice in Week 1** (`data/README.md`).

In [ ]:
from humanization_tools import NONHUMAN_AB, HUMAN_FRAMEWORK, split_regions, parental_sequence

# --- The non-human (e.g., murine/chimeric) antibody you are humanizing (TEACHING placeholder) ---
# VERIFY/replace with your real published therapeutic antibody VH (and repeat for VL) in Week 1.
print("NON-HUMAN ANTIBODY (to humanize):", NONHUMAN_AB["name"], "chain", NONHUMAN_AB["chain"])
for k in ("FR1", "CDR1", "FR2", "CDR2", "FR3", "CDR3", "FR4"):
    print(f"  {k:5s}: {NONHUMAN_AB[k]}")

# --- The human germline framework you graft the CDRs ONTO (TEACHING placeholder) ---
# VERIFY/replace with the exact IMGT human germline you select (e.g., an IGHV3 family member).
print("\nHUMAN GERMLINE FRAMEWORK (graft target):", HUMAN_FRAMEWORK["name"])
for k in ("FR1", "FR2", "FR3", "FR4"):
    print(f"  {k:5s}: {HUMAN_FRAMEWORK[k]}")

parent = parental_sequence(NONHUMAN_AB)
print("\nparental VH sequence length:", len(parent), "aa  (this is the ΔΔG / humanness BASELINE)")

## The Vernier zone (why grafting costs affinity)

The **Vernier zone** is the set of framework residues that pack against and *position* the CDR loops
(Foote & Winter 1992). When CDR grafting replaces a Vernier residue with the human one, the CDR loop can
shift → lost affinity/stability. Those positions are the prime **back-mutation** candidates: restore the
parental (non-human) residue to rescue binding, paying a small humanness cost. `VERNIER_ZONE` below uses
per-region positions approximating the canonical set (teaching-grade — map true Kabat/IMGT Vernier
positions with ANARCI for a real run).

In [ ]:
from humanization_tools import VERNIER_ZONE
print("Vernier-zone positions (per framework region, 0-based — TEACHING approximation):")
for region, positions in VERNIER_ZONE.items():
    print(f"  {region}: {positions}")
print("\nThese are the framework residues that support the CDR loops — the prime back-mutation targets.")

## Humanization hello-world (mock backend, no GPU)

Graft the non-human CDRs onto the human framework, score humanness + the ΔΔG proxy, and list the Vernier
back-mutations the graft suggests. This proves the plumbing
(graft → humanness → ΔΔG → back-mutations) before any real run. **Every number below is SYNTHETIC —
never report mock numbers as real.**

In [ ]:
from humanization_tools import (graft_cdrs, vernier_backmutations, score_variants,
                                  humanness_score, ddg_predict)

# 1) Graft the non-human CDRs onto the human germline framework (no back-mutations yet).
graft = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, scheme="kabat", tool="mock")

# 2) Score humanness + the ΔΔG proxy (vs the parental sequence).
score_variants([graft], parent, tool="mock")

# 3) The Vernier back-mutations this graft suggests (restore parental residues at CDR-support sites).
bms = vernier_backmutations(graft, NONHUMAN_AB, HUMAN_FRAMEWORK)

print("variant_id        :", graft.variant_id, "(method:", graft.method + ")")
print("VH length         :", len(graft.sequence), "aa")
print("framework muts     :", graft.n_framework_mutations, "(human FR residues differing from parental)")
print("humanness (heur)  : OASis-like", graft.oasis_like, "| T20-like", graft.t20_like,
      "| germline", graft.germline_id)
print("ΔΔG proxy (heur)  :", graft.ddg_kcal_mol, "(>0 = destabilizing; NOT real kcal/mol)")
print("synthetic         :", graft.synthetic, "->", graft.notes)
print("\nVernier back-mutation suggestions:")
for b in bms:
    print("  ", b["token"], "-", b["rationale"])

## A first look at the trade-off

Even at hello-world scale you can see the tension. Compare the **bare graft** (maximally human, costly)
to a graft **with Vernier back-mutations** (a little less human, more stable). Notebook 04 turns this
into the full trade-off figure across many variants and against the resurfacing alternative. **Mock
numbers are SYNTHETIC** — the *shape* of the trade-off is the teaching point, not the values.

In [ ]:
# Re-graft WITH the Vernier back-mutations applied — the affinity/stability rescue.
tokens = tuple(b["token"] for b in bms)
graft_bm = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, back_mutations=tokens, tool="mock")
score_variants([graft_bm], parent, tool="mock")

print(f"{'variant':28s} {'humanness':>10s} {'ΔΔG proxy':>10s} {'back-muts':>10s}")
print(f"{graft.variant_id:28s} {graft.oasis_like:>10} {graft.ddg_kcal_mol:>10} {len(graft.back_mutations):>10}")
print(f"{graft_bm.variant_id:28s} {graft_bm.oasis_like:>10} {graft_bm.ddg_kcal_mol:>10} {len(graft_bm.back_mutations):>10}")
print("\nExpected DIRECTION (SYNTHETIC values): back-mutations trade a little humanness for lower ΔΔG.")
print("This humanness<->stability trade-off IS the deliverable — quantify it in nb 04.")

## D0 checklist
- [ ] 1-page **problem statement**: the **non-human antibody** (verified, published), the **human
      germline framework(s)** you will graft onto, the humanization **strategy** (grafting vs
      resurfacing vs germline-content), and **measurable** success criteria (target humanness band +
      acceptable ΔΔG / retained-binding bar).
- [ ] Verified your antibody sequence + germline framework choice (IMGT/OAS); placeholders replaced.
- [ ] Metric table understood, including the "does not mean" column and that humanness + ΔΔG here are
      **heuristics**, not validated tools.
- [ ] Mock humanization hello-world run; graft + Vernier back-mutations + SYNTHETIC humanness/ΔΔG printed.
- [ ] `LOG.md` entry (seed, what you ran).

**Next:** `02_generate.ipynb` — graft CDRs onto candidate frameworks → variants CSV (mock now; real
AbLang/IgFold on Colab T4).

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — graft CDRs onto candidate human frameworks → variants CSV

**Standard slot:** *design campaign.* **For Project 16 this means:** run the humanization campaign —
graft the non-human CDRs onto **several candidate human germline frameworks**, generate variants with
and without **Vernier back-mutations** (and a resurfacing variant as the `[extension]` alternative),
score humanness + the ΔΔG proxy, and write a variants CSV (D2).

**Compute reality (be honest):** this project is **light** — humanness scoring, an IgFold/ImmuneBuilder
Fv model, and ΔΔG proxies are **free-tier Colab T4 friendly**. No A100 needed. This notebook runs on the
**mock** backend so the plumbing executes anywhere; switch each `tool="mock"` to the real backend
(AbLang / IgFold / FoldX-Rosetta) on Colab when ready.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (rule 4)

Tools change. Before a real run, confirm the pinned upstream repos still exist and **pin the exact
commit** you used (put it in `LOG.md`). This HTTP-checks the URLs; it does not install anything. Mark
the OASis/Hu-mAb/T20 humanness backends as **"verify current public release/host"** — they move.

In [ ]:
import requests

# Pinned upstreams for the humanization family (pin the COMMIT you actually use — these move).
UPSTREAMS = {
    "AbLang (antibody LM — humanness + restoration)": "https://github.com/oxpig/AbLang",
    "ImmuneBuilder / IgFold-style Fv (for ΔΔG model)": "https://github.com/oxpig/ImmuneBuilder",
    "ProteinMPNN (framework optimization [extension])": "https://github.com/dauparas/ProteinMPNN",
    "BioPhi (OASis / Hu-mAb humanness — VERIFY host)": "https://github.com/Merck/BioPhi",
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"[{r.status_code}] {name}\n        {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e}\n        {url}")
print("\nNOTE: OASis/Hu-mAb (BioPhi) + T20 server — VERIFY the current public release/host at course")
print("start and pin it (MANUAL.md §2). FoldX/Rosetta for ΔΔG are licensed — see MANUAL.md §2.")
print("Pin the exact COMMIT/tag of each tool in LOG.md before any real campaign.")

## Campaign parameters — candidate human frameworks

The CDRs (binding) are **fixed**; the **framework** is the design variable. Real humanization tries
**several candidate human germline frameworks** (the closest human germlines to the parental, by V/J
gene) and keeps the one that best balances humanness and stability. Here we mock a small panel of
candidate frameworks; on a real run, pull the actual IMGT germlines you chose in notebook 01.

In [ ]:
from humanization_tools import NONHUMAN_AB, HUMAN_FRAMEWORK, parental_sequence

parent = parental_sequence(NONHUMAN_AB)

# Candidate human germline frameworks. The first is the package placeholder; the others are small,
# deterministic perturbations standing in for DIFFERENT human germlines (e.g., IGHV1 vs IGHV3 vs IGHV4
# family members). REPLACE with the real IMGT germline FRs you selected in notebook 01.
def _variant_framework(base, name, swaps):
    fw = dict(base); fw["name"] = name
    for region, (pos, aa) in swaps.items():
        s = list(fw[region])
        if 0 <= pos < len(s):
            s[pos] = aa
        fw[region] = "".join(s)
    return fw

CANDIDATE_FRAMEWORKS = [
    HUMAN_FRAMEWORK,  # human_IGHV_teaching_placeholder
    _variant_framework(HUMAN_FRAMEWORK, "human_IGHV1-like_teaching", {"FR1": (5, "Q"), "FR3": (10, "K")}),
    _variant_framework(HUMAN_FRAMEWORK, "human_IGHV4-like_teaching", {"FR2": (3, "Q"), "FR3": (5, "N")}),
]
print("candidate human frameworks (VERIFY/replace with real IMGT germlines):")
for fw in CANDIDATE_FRAMEWORKS:
    print("  -", fw["name"])
print("\nTOOL = 'mock' (switch to 'ablang' on Colab T4 for real humanness-aware grafting)")

## Run the campaign (mock) → variants

For **each** candidate framework we generate three variants:
1. **bare CDR graft** (maximally human, expect high ΔΔG),
2. **graft + Vernier back-mutations** (rescue stability at a small humanness cost),
3. a **resurfacing** variant `[extension]` (surface FR residues only — fewer mutations, less human).

We also add the two mandatory **controls** the validation plan needs:
- the **parental** antibody (non-human; humanness floor, stability ceiling), and
- an **over-humanized decoy** (humanize aggressively *including* the Vernier zone — high humanness,
  expected to LOSE binding/stability; the negative control for "over-humanization").

In [ ]:
import pandas as pd
from humanization_tools import (graft_cdrs, resurface, vernier_backmutations, score_variants,
                                  HumanizedVariant)

variants = []
for fw in CANDIDATE_FRAMEWORKS:
    # 1) bare graft
    g = graft_cdrs(NONHUMAN_AB, fw, scheme="kabat", tool="mock",
                   variant_id=f"EXAMPLE_DATA_{fw['name']}_graft")
    # 2) graft + Vernier back-mutations
    bms = vernier_backmutations(g, NONHUMAN_AB, fw)
    tokens = tuple(b["token"] for b in bms)
    g_bm = graft_cdrs(NONHUMAN_AB, fw, back_mutations=tokens, tool="mock",
                      variant_id=f"EXAMPLE_DATA_{fw['name']}_graft_BM")
    variants.extend([g, g_bm])

# 3) resurfacing alternative [extension]
veneer = resurface(NONHUMAN_AB, tool="mock", variant_id="EXAMPLE_DATA_resurface")
variants.append(veneer)

# --- Controls ---
# parental (non-human) control: humanness floor + stability ceiling (ΔΔG = 0 by definition).
parental_ctrl = HumanizedVariant(
    variant_id="EXAMPLE_DATA_parental_control", sequence=parent, method="parental", tool="mock",
    cdr1=NONHUMAN_AB["CDR1"], cdr2=NONHUMAN_AB["CDR2"], cdr3=NONHUMAN_AB["CDR3"],
    synthetic=True, notes=["control: parental non-human antibody"])
# over-humanized decoy: graft AND humanize the Vernier zone too (no back-mutations) — expected to lose
# binding/stability. The negative control for over-humanization.
decoy = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, back_mutations=(), tool="mock",
                   variant_id="EXAMPLE_DATA_over_humanized_decoy")
decoy.method = "over_humanized_decoy"
decoy.notes.append("control: over-humanized decoy (no Vernier rescue) — expected to lose binding")
variants.extend([parental_ctrl, decoy])

score_variants(variants, parent, tool="mock")
print(f"generated {len(variants)} variants (grafts + back-mutated + resurface + 2 controls)")

In [ ]:
rows = [v.as_row() for v in variants]
camp = pd.DataFrame(rows)
cols = ["variant_id", "method", "tool", "parental", "human_framework", "scheme",
        "cdr1", "cdr2", "cdr3", "n_framework_mutations",
        "oasis_like", "t20_like", "germline_id", "ddg_kcal_mol", "synthetic"]
camp = camp[[c for c in cols if c in camp.columns]]
camp["cdr3_len"] = camp["cdr3"].str.len()
camp["n_back_mutations"] = [len(v.back_mutations) for v in variants]
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape)
print("SYNTHETIC?", bool(camp["synthetic"].all()), "(mock => all numbers are EXAMPLE_DATA)")
camp[["variant_id", "method", "n_framework_mutations", "n_back_mutations",
      "oasis_like", "ddg_kcal_mol"]]

## Quick campaign sanity look — the trade-off is already visible

Before filtering, eyeball the two axes that matter: **humanness** vs the **ΔΔG proxy**, coloured by
method. The parental control sits at low humanness / zero ΔΔG; bare grafts at high humanness / high ΔΔG;
back-mutated grafts and resurfacing in between. On **mock** these are SYNTHETIC and only show the
plumbing; on a real run this is your headline figure.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(camp["n_framework_mutations"], bins=10)
ax[0].set_title("framework mutations / variant"); ax[0].set_xlabel("# FR residues changed")
# humanness vs ΔΔG scatter, coloured by method
for method, sub in camp.groupby("method"):
    ax[1].scatter(sub["oasis_like"], sub["ddg_kcal_mol"], label=method, s=40)
ax[1].set_xlabel("humanness (OASis-like, heuristic)")
ax[1].set_ylabel("ΔΔG proxy (>0 destabilizing)")
ax[1].set_title("humanness vs ΔΔG (SYNTHETIC mock)")
ax[1].legend(fontsize=7)
plt.suptitle("Campaign pool — SYNTHETIC (mock) distributions; for plumbing only")
plt.tight_layout(); plt.savefig("results/campaign_distributions.png", dpi=150); plt.show()
print("Reminder: mock values are SYNTHETIC — real shape comes from OASis/Hu-mAb + FoldX/Rosetta.")

## D2 checklist
- [ ] `results/campaign.csv`: the variant pool (method, framework, CDRs, humanness + ΔΔG proxy, back-
      mutation count), one row per variant — including the **parental** and **over-humanized decoy**
      controls.
- [ ] Several **candidate human germline frameworks** tried (not just one).
- [ ] Version-verify cell run; exact tool **commits** pinned in `LOG.md`.
- [ ] Design log: parental antibody, candidate frameworks, scheme (Kabat/Chothia/IMGT), seed,
      tool/version, runtime.
- [ ] (Real run) humanness via OASis/Hu-mAb/T20/AbLang + ΔΔG via FoldX/Rosetta on an IgFold model;
      note that grafting typically LOSES affinity/stability and needs back-mutations.
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — the shared antibody filter (`design_type="antibody"`).

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (`design_type="antibody"`)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 16** you map your humanized variants onto `fp.Design` objects, run the pipeline with
the **antibody** cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12), and report survival (D3 pt 1).
We also fold in the humanization-specific axes (humanness floor + ΔΔG ceiling) as a physics-layer gate.

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"antibody"` cutoffs; humanization variants are single-domain/Fv-style, so the antibody cutoffs apply.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)
print("\nUsing design_type='antibody':", fp.DEFAULT_CUTOFFS["antibody"])

## Build `fp.Design` objects from the campaign

The filter operates on `fp.Design` records. For a humanization variant the structure-confidence fields
(`plddt`, `scrmsd`, `pae_interaction`) come from an IgFold/AF2 model of the Fv — on the **mock** backend
those are not produced, so we synthesize *clearly-SYNTHETIC* placeholder confidence values from the
deterministic hash so the confidence layers have something to act on (the plumbing point). The
**humanness** and **ΔΔG proxy** are the real axes of this project: we map humanness onto `solubility`
(so Layer 3 / physics gates on a humanness floor) and stash ΔΔG + humanness + method in `extra` so they
ride along into the ranked CSV.

In [ ]:
import hashlib

def _synthetic_confidence(variant_id, method):
    """SYNTHETIC structure-confidence placeholders for the mock path (NOT real predictions).

    On a real run these come from IgFold/ImmuneBuilder + AF2 of the Fv. The over-humanized decoy is
    given deliberately weaker confidence to mimic 'over-humanization broke the fold' — purely to show
    the filter discriminating; still SYNTHETIC."""
    h = int(hashlib.sha256(str(variant_id).encode()).hexdigest(), 16)
    plddt = 72 + (h % 22)                       # 72-93
    scrmsd = round(1.0 + (h % 180) / 100.0, 3)  # 1.0-2.8 Å
    pae = 6 + (h % 7)                           # 6-12
    if method == "over_humanized_decoy":
        plddt -= 10; scrmsd += 1.2; pae += 4    # nudge the decoy toward failing (SYNTHETIC)
    return float(plddt), float(scrmsd), float(pae)

camp = pd.read_csv("results/campaign.csv")

designs = []
for _, r in camp.iterrows():
    plddt, scrmsd, pae = _synthetic_confidence(r["variant_id"], r.get("method"))
    designs.append(fp.Design(
        design_id=str(r["variant_id"]),
        sequence="",                       # full VH not needed for the confidence layers
        design_type="antibody",
        plddt=plddt, scrmsd=scrmsd, pae_interaction=pae,
        # map humanness onto solubility so Layer 3 (physics) gates on a humanness FLOOR:
        solubility=r.get("oasis_like"),
        extra={"method": r.get("method"), "oasis_like": r.get("oasis_like"),
               "t20_like": r.get("t20_like"), "ddg_kcal_mol": r.get("ddg_kcal_mol"),
               "n_framework_mutations": r.get("n_framework_mutations"),
               "n_back_mutations": r.get("n_back_mutations"),
               "synthetic": bool(r.get("synthetic", False))},
    ))
print(len(designs), "fp.Design objects built (design_type='antibody')")
print("NOTE: structure-confidence values on the mock path are SYNTHETIC placeholders.")

## Run the pipeline + report

`run_pipeline(..., design_type="antibody")` applies the layers in order with the antibody cutoffs and
returns a ranked DataFrame; `report()` prints the **survival-at-each-layer** accounting and saves the
ranked CSV + figure. We run Layers 1 + 3 here (self-consistency + physics, where physics = a humanness
floor via `min_solubility`). Layer 2 (orthogonal predictor agreement) needs a second predictor — wire
ESMFold/IgFold scRMSD in for the real run. **On mock data the survivors are SYNTHETIC** — the point is
the plumbing and the honest accounting.

In [ ]:
# Layer 3 physics gate: require humanness (mapped to `solubility`) >= a floor. Justify the floor in
# your report against real OASis/Hu-mAb distributions; 0.6 is a teaching default for the proxy.
HUMANNESS_FLOOR = 0.6

df_ranked = fp.run_pipeline(
    designs, design_type="antibody", use_layers=(1, 3),
    cutoffs={**fp.DEFAULT_CUTOFFS["antibody"]},
)
# physics_filter() uses min_solubility=-1.0 by default; re-run the physics layer explicitly with the
# humanness floor so the gate is meaningful for this project:
for d in designs:
    d.layers_passed = 0; d.notes = []
survival = {"L1": 0, "L3": 0}
for d in designs:
    if fp.self_consistency(d, fp.DEFAULT_CUTOFFS["antibody"]):
        survival["L1"] += 1
        if fp.physics_filter(d, fp.DEFAULT_CUTOFFS["antibody"], min_solubility=HUMANNESS_FLOOR):
            survival["L3"] += 1
df_ranked = fp.rank_designs(designs)
import pandas as pd
df_ranked = pd.DataFrame([fp.asdict(d) for d in df_ranked])
df_ranked.attrs["survival"] = survival
df_ranked.attrs["n_total"] = len(designs)

top = fp.report(df_ranked, top_n=15, save_prefix="results/proj16")
print("\nranked CSV -> results/proj16_ranked.csv ; survival figure -> results/proj16_survival.png")
print(f"(Layer 3 humanness floor = {HUMANNESS_FLOOR} on the OASis-like proxy.)")
top

## Hit-rate accounting (report the rate, not the cherry)

Survival = how many of the variant pool pass each layer. For humanization the meaningful "hit" is a
variant that is **human enough** (passes the humanness floor) **and** structurally self-consistent —
*and*, in the full study, has an acceptable ΔΔG. Expect the **over-humanized decoy** to fail and the
**parental control** to fail the humanness floor (it is non-human by definition) — that is the filter
behaving correctly, not a bug.

In [ ]:
n_total = df_ranked.attrs.get("n_total", len(df_ranked))
survival = df_ranked.attrs.get("survival", {})
print(f"Generated: {n_total} variants")
for layer, n in survival.items():
    print(f"  {layer}: {n} survivors ({100*n/max(n_total,1):.1f}%)")
print("\nlayers_passed distribution:")
if "layers_passed" in df_ranked:
    print(df_ranked["layers_passed"].value_counts().sort_index())

# Show how each control fared (sanity: decoy should struggle; parental should fail the humanness floor).
print("\ncontrols / methods:")
view = df_ranked.copy()
view["method"] = [e.get("method") for e in view["extra"]]
print(view[["design_id", "method", "layers_passed", "scrmsd", "plddt"]].to_string(index=False))
print("\nReminder: mock survivors are SYNTHETIC. Real survival comes from OASis/Hu-mAb + IgFold + FoldX.")

## D3 (part 1) checklist
- [ ] `results/proj16_ranked.csv` produced by the **shared** module with `design_type="antibody"`.
- [ ] Survival-at-each-layer reported (the survival figure saved), with a justified **humanness floor**.
- [ ] Mapping assumptions written down (humanness → `solubility`; ΔΔG/method stashed in `extra`).
- [ ] Controls behave sanely: over-humanized decoy struggles; parental fails the humanness floor.
- [ ] Honest hit-rate accounting; humanness + ΔΔG flagged as heuristics on the mock path.

**Next:** `04_validate.ipynb` — the humanness↔stability trade-off, Vernier back-mutations, and the
grafting-vs-resurfacing comparison.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — humanness↔stability trade-off, Vernier back-mutations, grafting vs resurfacing

**Standard slot:** *validate (in silico).* **For Project 16 the core analyses are:** (1) the
**humanness ↔ stability (ΔΔG) trade-off** across variants, (2) the effect of **Vernier-zone
back-mutations** (rescue stability at a humanness cost), and (3) the **CDR-grafting vs resurfacing**
comparison `[extension]` (D3 pt 2). This is what makes the project a *study*, not a demo.

Needs `results/campaign.csv`. All metrics on the mock backend are **SYNTHETIC** — the figures here
demonstrate the analysis; real numbers come from OASis/Hu-mAb + FoldX/Rosetta on an IgFold Fv model.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · The humanness ↔ stability trade-off `[core]`

The headline result. Plot **humanness** (x) against the **ΔΔG proxy** (y, lower = more stable) for every
variant, coloured by method. The frontier you care about is the lower-right: **high humanness AND low
ΔΔG**. The parental control anchors low-humanness/zero-ΔΔG; bare grafts sit high-humanness/high-ΔΔG;
back-mutated grafts and resurfacing trade along the curve. **Mock values are SYNTHETIC** — the *shape*
and the *reasoning* are the deliverable.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")

fig, ax = plt.subplots(figsize=(7, 4.5))
markers = {"cdr_graft": "o", "resurface": "s", "parental": "*", "over_humanized_decoy": "X"}
for method, sub in camp.groupby("method"):
    ax.scatter(sub["oasis_like"], sub["ddg_kcal_mol"],
               marker=markers.get(method, "o"), s=70, label=method)
    for _, r in sub.iterrows():
        ax.annotate(str(r["variant_id"]).replace("EXAMPLE_DATA_", ""),
                    (r["oasis_like"], r["ddg_kcal_mol"]), fontsize=6, alpha=0.7)
ax.set_xlabel("humanness (OASis-like, heuristic — higher = more human)")
ax.set_ylabel("ΔΔG proxy (a.u.; >0 = destabilizing — lower = more stable)")
ax.set_title("Humanness vs stability trade-off — SYNTHETIC (mock)\nwant: lower-right (human AND stable)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("results/tradeoff.png", dpi=150); plt.show()
print("Reminder: SYNTHETIC. On a real run use OASis/Hu-mAb humanness + FoldX/Rosetta ΔΔG.")

## 2 · Vernier-zone back-mutations — the rescue `[core]`

Back-mutations restore parental residues at framework positions that support the CDRs. They should
**lower ΔΔG** (regain stability/affinity) at a **small humanness cost**. Compare each bare graft to its
back-mutated counterpart; report the ΔΔG regained per humanness lost. This is the core decision a
humanization campaign makes: *which* Vernier residues to restore.

In [ ]:
from humanization_tools import (NONHUMAN_AB, HUMAN_FRAMEWORK, graft_cdrs, vernier_backmutations,
                                  score_variants, parental_sequence)

parent = parental_sequence(NONHUMAN_AB)

# Build the back-mutation ladder: graft, then add Vernier back-mutations one at a time, and watch
# humanness fall slightly while ΔΔG falls (rescue). SYNTHETIC values — the trend is the teaching point.
g = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, tool="mock")
bms = vernier_backmutations(g, NONHUMAN_AB, HUMAN_FRAMEWORK)
tokens = [b["token"] for b in bms]

ladder = []
for k in range(0, len(tokens) + 1):
    v = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, back_mutations=tuple(tokens[:k]), tool="mock",
                   variant_id=f"EXAMPLE_DATA_BMladder_{k}")
    score_variants([v], parent, tool="mock")
    ladder.append(dict(n_back_mutations=k, humanness=v.oasis_like, ddg_proxy=v.ddg_kcal_mol))
ladder_df = pd.DataFrame(ladder)
print("Back-mutation ladder (SYNTHETIC — direction is the teaching point):")
print(ladder_df.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(6.5, 3.6))
ax1.plot(ladder_df["n_back_mutations"], ladder_df["humanness"], "o-", color="tab:blue", label="humanness")
ax1.set_xlabel("# Vernier back-mutations applied"); ax1.set_ylabel("humanness (heuristic)", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(ladder_df["n_back_mutations"], ladder_df["ddg_proxy"], "s--", color="tab:red", label="ΔΔG proxy")
ax2.set_ylabel("ΔΔG proxy (lower = more stable)", color="tab:red")
plt.title("Vernier back-mutations: stability rescue vs humanness cost (SYNTHETIC)")
plt.tight_layout(); plt.savefig("results/backmutation_ladder.png", dpi=150); plt.show()

## 3 · CDR grafting vs resurfacing `[extension]`

The two humanization strategies make a different bet. **CDR grafting** changes the whole framework
(high humanness, high ΔΔG risk, needs back-mutations). **Resurfacing** changes only surface framework
residues (low humanness gain, low ΔΔG risk). Compare them head-to-head on the same parental antibody:
which reaches your target humanness band at the lower stability cost? There is no universal winner — it
depends on the antibody and your humanness/ΔΔG bars.

In [ ]:
from humanization_tools import resurface

graft = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, tool="mock", variant_id="EXAMPLE_DATA_graft")
graft_bm = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, back_mutations=tuple(tokens), tool="mock",
                      variant_id="EXAMPLE_DATA_graft_BM")
veneer = resurface(NONHUMAN_AB, tool="mock", variant_id="EXAMPLE_DATA_resurface")
score_variants([graft, graft_bm, veneer], parent, tool="mock")

compare = pd.DataFrame([
    dict(strategy="CDR graft (bare)", humanness=graft.oasis_like, ddg_proxy=graft.ddg_kcal_mol,
         fr_muts=graft.n_framework_mutations),
    dict(strategy="CDR graft + Vernier BM", humanness=graft_bm.oasis_like, ddg_proxy=graft_bm.ddg_kcal_mol,
         fr_muts=graft_bm.n_framework_mutations),
    dict(strategy="Resurfacing", humanness=veneer.oasis_like, ddg_proxy=veneer.ddg_kcal_mol,
         fr_muts=veneer.n_framework_mutations),
])
print("Grafting vs resurfacing (SYNTHETIC mock):")
print(compare.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.scatter(compare["humanness"], compare["ddg_proxy"], s=90)
for _, r in compare.iterrows():
    ax.annotate(r["strategy"], (r["humanness"], r["ddg_proxy"]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("humanness (heuristic)"); ax.set_ylabel("ΔΔG proxy (lower = more stable)")
ax.set_title("Grafting vs resurfacing — SYNTHETIC (want lower-right)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.savefig("results/grafting_vs_resurfacing.png", dpi=150); plt.show()
print("Reminder: SYNTHETIC. No universal winner — report the trade-off for YOUR antibody with real tools.")

## 4 · Immunogenicity-risk summary (qualitative) `[core]`

Beyond the single humanness number, summarize the residual immunogenicity risk: how many framework
residues remain non-human, whether any back-mutations re-introduced non-human residues at exposed
positions, and whether known T-cell-epitope-prone motifs persist (a real run would add a T-cell epitope
predictor). Frame it as **risk**, not a guarantee — humanness scores correlate with, but do not prove,
low ADA.

In [ ]:
risk = []
for v_id, method, n_fr, n_bm, hum in zip(
        camp["variant_id"], camp["method"], camp["n_framework_mutations"],
        camp["n_back_mutations"], camp["oasis_like"]):
    band = "LOW" if hum >= 0.7 else ("MODERATE" if hum >= 0.5 else "HIGH")
    risk.append(dict(variant_id=v_id, method=method, humanness=hum,
                     residual_nonhuman_fr=int(n_fr) if pd.notna(n_fr) else None,
                     vernier_back_mutations=int(n_bm) if pd.notna(n_bm) else None,
                     immunogenicity_risk_band=band))
risk_df = pd.DataFrame(risk).sort_values("humanness", ascending=False)
risk_df.to_csv("results/immunogenicity_risk_summary.csv", index=False)
print("wrote results/immunogenicity_risk_summary.csv (SYNTHETIC bands — confirm with real tools + a")
print("T-cell-epitope predictor; humanness != guaranteed low ADA)")
risk_df

## D3 (part 2) checklist
- [ ] **Humanness ↔ ΔΔG trade-off** figure (the headline) with the frontier reasoning.
- [ ] **Vernier back-mutation** ladder: ΔΔG regained vs humanness lost; which residues to restore.
- [ ] **Grafting vs resurfacing** comparison `[extension]`; state which wins for *your* antibody and why.
- [ ] **Immunogenicity-risk summary** (residual non-human content + risk band), framed as risk.
- [ ] Every mock number labelled SYNTHETIC; conclusions phrased as plumbing/teaching, not results.

**Next:** `05_validation_plan.ipynb` — the ELISA/SPR/DSF validation plan + controls (parental +
over-humanized decoy).

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — express, ELISA/SPR (binding), DSF (stability), controls

**Standard slot:** *validation plan.* **For Project 16 this means:** a wet-lab plan to test whether the
humanized variants **retain binding** (ELISA / SPR) and **stay stable** (DSF), plus an
**immunogenicity-risk summary**, with the two mandatory **controls** — the **parental** (non-human)
antibody and an **over-humanized decoy** (D4/D5).

This generates structured plan files and a costed-reagent stub; it runs with no GPU. The deliverable is
**variants + the trade-off analysis + the experiment** — a humanness score is a hypothesis until the
assays confirm retained binding and stability.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Why this experiment (not "we humanized it")

A humanized sequence is a **hypothesis**. A high humanness score does **not** prove low immunogenicity,
and grafting frequently **loses binding/stability** — that is exactly why the campaign produced *several*
variants (grafts, back-mutated grafts, resurfaced) plus controls. The experiment answers two questions
per variant: **does it still bind the antigen** (ELISA/SPR vs the parental) and **is it still stable**
(DSF Tm vs the parental)? The **over-humanized decoy** is the negative control that proves the assay can
detect over-humanization (it should lose binding and/or Tm).

## 1 · The binding + stability assay plan `[core]`

ELISA (fast yes/no binding), SPR/BLI (quantitative K_D), and DSF (thermal stability, Tm). The plan
generator records the stages, readouts, and controls so it is reproducible and gradable.

In [ ]:
import json, os

assay_plan = {
    "goal": "Test whether humanized variants RETAIN antigen binding and stability vs the parental.",
    "variants_under_test": "results/proj16_ranked.csv survivors (grafts, back-mutated grafts, resurfaced)",
    "expression": {
        "format": "Fab or full IgG (mammalian, e.g. ExpiCHO/HEK293) for binding/DSF; "
                  "scFv/Fab in E. coli periplasm acceptable for early ELISA triage",
        "note": "express each variant + BOTH controls under identical conditions",
    },
    "assays": [
        {"name": "ELISA", "measures": "qualitative antigen binding (retained vs lost)",
         "readout": "OD450 titration vs antigen-coated plate; EC50 estimate"},
        {"name": "SPR or BLI", "measures": "quantitative affinity (K_D, k_on, k_off)",
         "readout": "kinetics vs immobilized antigen; compare K_D to the parental"},
        {"name": "DSF (thermal shift)", "measures": "stability (melting temperature Tm)",
         "readout": "SYPRO-Orange Tm; compare Tm to the parental (humanization often lowers Tm)"},
    ],
    "decision": "KEEP variants that retain binding (K_D within an agreed fold of parental) AND keep Tm "
                "within an agreed margin; these balance humanness and developability.",
    "controls": {
        "positive_parental": "the PARENTAL non-human antibody — defines retained-binding (best K_D) and "
                             "the stability ceiling (Tm); every humanized variant is judged against it",
        "negative_over_humanized_decoy": "the OVER-HUMANIZED DECOY (humanized including the Vernier zone, "
                                         "no rescue) — expected to LOSE binding and/or Tm; proves the "
                                         "assay detects over-humanization",
        "negative_isotype": "an irrelevant isotype-matched antibody — no specific antigen binding",
    },
    "expectation": "Humanization commonly costs some affinity/stability; Vernier back-mutated and "
                   "resurfaced variants should recover more than the over-humanized decoy.",
}
os.makedirs("results", exist_ok=True)
with open("results/validation_plan.json", "w") as fh:
    json.dump(assay_plan, fh, indent=2)
print("wrote results/validation_plan.json")
for a in assay_plan["assays"]:
    print(f"  {a['name']:12s} -> {a['measures']}")
print("\ncontrols:")
for k, v in assay_plan["controls"].items():
    print(f"  {k}: {v}")

## 2 · Immunogenicity-risk summary `[core]`

The clinical motivation. Summarize each variant's residual immunogenicity risk: humanness band, residual
non-human framework content, any non-human residues re-introduced by back-mutations at exposed positions,
and (real run) predicted T-cell epitopes. State it as **risk**, paired with the *retained-binding* result
— a maximally-human variant that does not bind is useless. The honest framing: humanness reduces, but
does not eliminate, ADA risk; only a clinical immunogenicity assessment is definitive.

In [ ]:
import json
immuno = {
    "summary": "Per-variant residual-immunogenicity risk, paired with the retained-binding result.",
    "inputs": "results/immunogenicity_risk_summary.csv (humanness band + residual non-human FR content)",
    "axes": [
        "humanness band (OASis/Hu-mAb/T20 — use the REAL tools to report)",
        "residual non-human framework residues (count + exposed?)",
        "non-human residues re-introduced by Vernier back-mutations (exposed positions are higher risk)",
        "predicted T-cell epitopes (add a real predictor, e.g. NetMHCII-style, for a reportable claim)",
    ],
    "honesty": "Humanness scores correlate with, but do NOT prove, low ADA. Only a clinical "
               "immunogenicity assessment is definitive. Pair every humanness claim with the assay result.",
    "decision_rule": "Prefer the MOST human variant that still PASSES the binding (ELISA/SPR) and "
                     "stability (DSF) bars vs the parental. Do not chase humanness past loss of function.",
}
with open("results/immunogenicity_risk_plan.json", "w") as fh:
    json.dump(immuno, fh, indent=2)
print(json.dumps(immuno, indent=2))

## 3 · Controls (mandatory) — parental + over-humanized decoy `[core]`

Controls are non-negotiable, even in the plan. The **parental** (non-human) antibody is the
positive/retained-binding reference and the stability ceiling. The **over-humanized decoy** is the
negative control proving the assay detects over-humanization. An **isotype** control rules out
non-specific binding.

In [ ]:
controls = {
    "positive_parental": "PARENTAL non-human antibody — best binding + highest Tm; the benchmark every "
                         "humanized variant is compared against (retained binding = within agreed fold).",
    "negative_over_humanized_decoy": "OVER-HUMANIZED DECOY — humanized including the Vernier zone with no "
                                     "back-mutation rescue; expected to LOSE binding and/or Tm. If it "
                                     "still binds well, your assay (or your Vernier set) needs revisiting.",
    "negative_isotype": "irrelevant isotype-matched antibody — controls for non-specific binding.",
    "why_decoy": "Without an over-humanized decoy you cannot show your assay distinguishes 'human enough' "
                 "from 'too humanized to bind' — the central risk of this project.",
}
with open("results/controls.json", "w") as fh:
    json.dump(controls, fh, indent=2)
print(json.dumps(controls, indent=2))

## 4 · Costed reagent + timeline stub `[extension]`

A skeleton the student fills with real quotes/timelines. Numbers below are **EXAMPLE_DATA placeholders**
(not real quotes) — replace with vendor quotes in your D4 plan.

In [ ]:
import pandas as pd
plan_items = pd.DataFrame([
    dict(item="Gene synthesis (variants + 2 controls)", purpose="express each variant", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Mammalian expression (ExpiCHO/HEK)", purpose="Fab/IgG production", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Protein A/affinity purification", purpose="purify antibodies", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Recombinant antigen (ELISA/SPR)", purpose="binding assays", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="ELISA reagents + plates", purpose="binding triage", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="SPR/BLI chip time", purpose="K_D vs parental", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="DSF (SYPRO-Orange, qPCR plate)", purpose="Tm stability", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="(Real run) humanness tools / T-cell epitope predictor", purpose="immunogenicity risk", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
])
plan_items.to_csv("results/experimental_plan.csv", index=False)
print("wrote results/experimental_plan.csv (EXAMPLE_DATA placeholders — fill with real quotes)")
plan_items

## Responsible research (state in the plan)

This is a **therapeutic antibody** project — **reducing immunogenicity** (ADA risk) of a non-human
therapeutic antibody so it is safer/more effective in patients. This is a defensive, low-dual-use aim.
In scope: humanizing therapeutic/diagnostic antibodies. Out of scope: enhancing pathogen
transmissibility/virulence, toxins, or any design intended to cause harm. Any real gene-synthesis order
must go through a biosecurity-screening provider (IGSC member); wet-lab work requires institutional
biosafety/ethics approval. Do not overstate a humanness score as a proven low-ADA outcome — only a
clinical immunogenicity assessment is definitive. See `MASTER_BLUEPRINT.md §7`.

## D4 / D5 checklist
- [ ] `results/validation_plan.json`: ELISA + SPR/BLI + DSF plan with stages, readouts, decision rule.
- [ ] `results/immunogenicity_risk_plan.json` + `immunogenicity_risk_summary.csv`: residual risk,
      paired with the retained-binding result, framed as risk (not a guarantee).
- [ ] `results/controls.json`: the **parental** + **over-humanized decoy** (+ isotype) controls, with
      why the decoy is essential.
- [ ] Costed reagent + timeline stub (EXAMPLE_DATA → real quotes).
- [ ] Responsible-research framing stated (immunogenicity reduction; low dual-use).
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — this project follows the **antibody-family pattern** (Project 17's template): the
`design_type="antibody"` filter hand-off, a deterministic mock, and a controlled validation plan.